This file is used to generate optical flow score for each videos. 

## CONFIGURATIONS

In [ ]:
CONFIG = {
    "video_file_path": "data\\videos",
    "num_videos": 250,
    "save_path": "data\\metadata\\optical_flow_score.npy"
}

## Calculate optical flow for one video

In [3]:
import cv2
import numpy as np

def calculate_optical_flow_for_video(video_path: str) -> float:
    """
    Calculates the average optical flow magnitude for a single video file.

    This function reads a video, computes the dense optical flow between
    consecutive frames using the Farnebäck algorithm, and returns the
    average magnitude of flow across all frames.

    Args:
        video_path (str): The path to the video file.

    Returns:
        float: The average optical flow score for the video. Returns 0.0
               if the video cannot be opened or has fewer than two frames.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Warning: Could not open video file {video_path}")

    # Read the first frame
    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError(f"Warning: Could not read the first frame of {video_path}")
    
    # Convert the first frame to grayscale for optical flow calculation
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    total_flow_magnitude = 0.0
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate dense optical flow using Farnebäck method
        # Parameters can be tuned, but defaults are often a good start.
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Calculate the magnitude (length) of the flow vectors
        # flow is a 3D array with shape (height, width, 2)
        # flow[..., 0] is the horizontal component (u)
        # flow[..., 1] is the vertical component (v)
        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        
        # Calculate the average magnitude for the current frame pair
        frame_avg_magnitude = np.mean(magnitude)
        
        total_flow_magnitude += frame_avg_magnitude
        frame_count += 1
        
        # Update the previous frame
        prev_gray = gray

    cap.release()

    if frame_count == 0:
        raise ValueError(f"Warning: Could not read any frames from {video_path}")
        
    # Return the average flow magnitude over the entire video
    video_avg_flow = total_flow_magnitude / frame_count
    return video_avg_flow

## calculate optical flow score for every videos

In [4]:
from tqdm import tqdm
import os

def generate_all_scores(video_directory: str, num_videos: int, output_path: str):
    """
    Generates optical flow scores for all videos in a directory and saves them.

    Assumes videos are named '1.mp4', '2.mp4', etc.

    Args:
        video_directory (str): The path to the directory containing video files.
        num_videos (int): The total number of videos to process.
        output_path (str): The path to save the resulting .npy file.
    """
    all_scores = []
    
    # Assuming videos are named 1.mp4, 2.mp4, ..., num_videos.mp4
    # Adjust the loop if your naming convention is different
    for i in tqdm(range(1, num_videos + 1), desc="Processing Videos"):
        video_file = f"{i}.mp4" # or ".gif", ".avi" etc.
        video_path = os.path.join(video_directory, video_file)

        if not os.path.exists(video_path):
            raise ValueError(f"Warning: Video file not found: {video_path}. Skipping.")

        score = calculate_optical_flow_for_video(video_path)
        all_scores.append(score)

    # Convert to numpy array and save
    scores_array = np.array(all_scores)
    np.save(output_path, scores_array)
    print(f"Successfully saved {len(all_scores)} optical flow scores to {output_path}")
    print(f"Sample scores: {scores_array[:10]}")

## Start!

In [5]:
generate_all_scores(CONFIG["video_file_path"], CONFIG["num_videos"], CONFIG["save_path"])

Processing Videos: 100%|██████████| 3/3 [02:19<00:00, 46.40s/it]

Successfully saved 3 optical flow scores to data\metadata\optical_flow_score.npy
Sample scores: [0.7449069  0.19424805 0.37836003]
